# FireSight Phase 3b - detector learning-curve pilot (Kaggle, T4)

**Purpose:** does more training data still buy recall at a fixed false-alarm rate?

**Base weights:** `yolo11s.pt` for all three runs (downloaded at runtime). Licence: **AGPL-3.0** (Ultralytics default) - commercial distribution may require an Ultralytics Enterprise Licence; record before shipping.

**Rules**
- Exactly **one GPU notebook at a time** (free quota ~30 h/week). The preflight cell refuses to continue if more than one GPU is visible.
- Selection data is `data/splits/val.jsonl` slices only. **bench_R, TEST and testbench100 are not in this dataset** (see `LEAKAGE_REPORT.json`).
- Every epoch is checkpointed to `/kaggle/working`; after a session restart re-run the train cell and it resumes from `last.pt`.
- Do **not** upload anything from here. The dataset is private.

**Estimated GPU hours (yolo11s @640, batch 64, AMP, T4)** - to be replaced by measured `images/s`:

| run | images | epochs | est. @60 img/s | est. @100 img/s |
|---|---:|---:|---:|---:|
| P10 | 5,898 | 15 | 0.5 h | 0.3 h |
| P25 | 14,747 | 15 | 1.1 h | 0.7 h |
| P50 | 29,493 | 15 | 2.2 h | 1.4 h |
| **pilot total** | | | **~3.8 h** | **~2.4 h** |

**Credentials:** never paste a Kaggle token into a notebook or the repo. `kaggle.json` / `KAGGLE_KEY` are read by the CLI only, from outside the repo.


In [ ]:
import json, os, subprocess, sys
import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x1 in the notebook settings"
n = torch.cuda.device_count()
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpus", n)
for i in range(n):
    print("gpu", i, torch.cuda.get_device_name(i))
assert n >= 1, "No GPU visible"
# Kaggle may expose 2 T4s in one session; training is pinned to device 0.
print(f"OK: {n} GPU(s) visible, training pinned to device 0. Run only ONE GPU notebook at a time.")


In [ ]:
# Pinned versions (torch stays as provided by the Kaggle GPU image for CUDA compatibility)
!pip install -q ultralytics==8.4.147 onnx==1.22.0 onnxruntime==1.30.0
import ultralytics, onnx, onnxruntime, cv2, numpy, PIL
print(json.dumps({"ultralytics": ultralytics.__version__, "onnx": onnx.__version__,
                  "onnxruntime": onnxruntime.__version__, "opencv": cv2.__version__,
                  "numpy": numpy.__version__, "pillow": PIL.__version__,
                  "torch": torch.__version__}, indent=2))


In [ ]:
import os, json, sys

# The dataset is uploaded as directory zips and Kaggle extracts them, so the layout is
# /kaggle/input/firesight-pilot/{images,labels,lists,code} + data.yaml.
DATASET = "/kaggle/input/firesight-pilot"
assert os.path.isdir(f"{DATASET}/images"), f"missing images dir; found {sorted(os.listdir(DATASET))}"
assert os.path.isdir(f"{DATASET}/labels") and os.path.isdir(f"{DATASET}/lists")
sys.path.insert(0, f"{DATASET}/code")

print(open(f"{DATASET}/manifest_summary.json").read())
print(open(f"{DATASET}/LEAKAGE_REPORT.json").read())
assert json.load(open(f"{DATASET}/LEAKAGE_REPORT.json"))["verdict"] == "PASS", "Leakage check failed - stop"
for f in ("train_10.txt", "train_25.txt", "train_50.txt", "val_early.txt", "val_cal.txt", "val_report.txt"):
    print(f, sum(1 for l in open(f"{DATASET}/lists/{f}") if l.strip()))
print("images:", len(os.listdir(f"{DATASET}/images")), "labels:", len(os.listdir(f"{DATASET}/labels")))


## 1. Train the three learning-curve points

Each run checkpoints every epoch. If a session dies, re-run this cell: it detects
`last.pt` and passes `--resume`. `P10n` (yolo11n) is an optional pipeline sanity arm.


In [ ]:
import os
FRACTIONS = ["10", "25", "50"]
for frac in FRACTIONS:
    out = f"{WORK}/runs/P{frac}"
    resume = "--resume" if os.path.exists(f"{out}/weights/last.pt") else ""
    cmd = (f"python {DATASET}/code/pilot_train.py --dataset {DATASET} --fraction {frac} "
           f"--model yolo11s.pt --epochs 15 --imgsz 640 --batch 64 --device 0 --workers 4 --amp "
           f"--out {out} {resume}")
    print("RUN:", cmd, flush=True)
    !{cmd}


## 2. Score each run on the val.jsonl report slice

Thresholds are calibrated on `val_cal` (val.jsonl) for **FPR <= 2% per negative image**,
then scored on the disjoint `val_report` slice with Wilson 95% CIs.


In [ ]:
from ultralytics import YOLO
from pathlib import Path

def read_list(name):
    # lists are relative to the dataset root (images/<file>) - resolve before predicting
    out = []
    for line in open(f"{DATASET}/lists/{name}"):
        line = line.strip()
        if not line:
            continue
        p = Path(line)
        out.append(p if p.is_absolute() else Path(DATASET) / p)
    return out

def predict_val(weights, imgsz=640):
    paths = read_list("val_cal.txt") + read_list("val_report.txt")
    model = YOLO(weights)
    rows = []
    for res in model.predict(paths, imgsz=imgsz, conf=0.01, verbose=False, stream=True):
        dets = []
        for b in res.boxes:
            cls = int(b.cls.item())
            dets.append({"label": "fire" if cls == 0 else "smoke",
                         "confidence": round(float(b.conf.item()), 6),
                         "box": [float(v) for v in b.xyxy[0].tolist()]})
        rows.append({"id": Path(res.path).stem, "detections": dets})
    out = f"{WORK}/preds_{Path(weights).parent.parent.name}.jsonl"
    with open(out, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")
    return out

for frac in FRACTIONS:
    weights = f"{WORK}/runs/P{frac}/weights/best.pt"
    if not os.path.exists(weights):
        print("missing", weights); continue
    preds = predict_val(weights)
    !python {DATASET}/code/pilot_eval.py --dataset {DATASET} --preds {preds} --model P{frac} --target-fpr 0.02 --out {WORK}/metrics_P{frac}.json


## 3. Summary table and learning curve

In [ ]:
import glob, json
import matplotlib.pyplot as plt

rows = []
for frac in FRACTIONS:
    rp, mp = f"{WORK}/runs/P{frac}/run_summary.json", f"{WORK}/metrics_P{frac}.json"
    if not (os.path.exists(rp) and os.path.exists(mp)):
        continue
    run, met = json.load(open(rp)), json.load(open(mp))
    c = met["splits"]["val"]["combined"]
    rows.append({"fraction": frac, "images": run["n_train"],
                 "images_per_s": run.get("last_epoch_images_per_s") or run["throughput"].get("images_per_s"),
                 "s_per_epoch": run.get("last_epoch_time_s"),
                 "recall": c["recall"], "ci": c["recall_95ci"],
                 "fpr_neg": met["splits"]["val"]["fpr_per_negative_image"]["fpr"],
                 "map50": met["splits"]["val"]["box_ap"]["fire"]["ap50"]})
print(f"{'run':6s} {'images':>7s} {'img/s':>7s} {'s/epoch':>8s} {'recall@FPR2%':>14s} {'95% CI':>18s} {'FPR/neg':>8s}")
for r in rows:
    ci = f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]" if r['ci'] else "-"
    print(f"P{r['fraction']:5s} {r['images']:7d} {str(r['images_per_s']):>7s} {str(r['s_per_epoch']):>8s} "
          f"{r['recall'] if r['recall'] is not None else float('nan'):14.4f} {ci:>18s} {str(r['fpr_neg']):>8s}")
json.dump(rows, open(f"{WORK}/pilot_summary.json", "w"), indent=2)

if rows:
    xs = [r["images"] for r in rows]; ys = [r["recall"] or 0 for r in rows]
    lo = [r["ci"][0] if r["ci"] else 0 for r in rows]; hi = [r["ci"][1] if r["ci"] else 0 for r in rows]
    plt.figure(figsize=(6, 4))
    plt.errorbar(xs, ys, yerr=[[y - l for y, l in zip(ys, lo)], [h - y for y, h in zip(ys, hi)]], marker="o")
    plt.xlabel("training images"); plt.ylabel("val recall @ FPR<=2% / negative")
    plt.title("FireSight pilot learning curve (val.jsonl)")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(f"{WORK}/learning_curve.png", dpi=160); plt.show()


In [ ]:
# ONNX export of the 50% run (deployment format; 10 MB for yolo11n, ~40 MB for yolo11s)
from ultralytics import YOLO
best = f"{WORK}/runs/P50/weights/best.pt"
if os.path.exists(best):
    print(YOLO(best).export(format="onnx", imgsz=640, opset=17, dynamic=False, simplify=False))
else:
    print("P50 best.pt missing; run the training cell first")


## 4. Decision rules (fill in, do not upload)

| outcome | rule | action |
|---|---|---|
| **GO** | recall rises from P10 to P50 by >= 5 pp with non-overlapping CIs, P50 beats the baseline at equal FPR, **and one bench_R DEV-only run per checkpoint (300 images) reproduces the ordering; SEALED untouched** | approve the full run; recompute full-run hours from measured `img/s` |
| **TRIM** | curve rises but projected full run exceeds ~20 GPU h or quota | fewer epochs / 25-50% data / yolo11n, then re-approve |
| **STOP** | P10 -> P50 flat within CI, or no run beats baseline at equal FPR | stop; diagnose label noise / task mismatch / augmentation before more training |

Report: `pilot_summary.json`, `learning_curve.png`, per-run `run_summary.json` and `metrics_P*.json`.
**Nothing is uploaded from this notebook.** Stop here and share the summary.
